# Open Insect ID v2 — Export to ONNX / INT8 / TFLite

CPU notebook. Loads the trained cascade, exports the trunk + the 4 heads to
ONNX, applies INT8 post-training quantization to the trunk, and converts to
`.tflite` for the Android app (heads ship as FP16 matrices). Attach the
checkpoint output-dataset from notebook 02.


In [ ]:
import os, subprocess, sys
from pathlib import Path

# --- Configure this to your repo -------------------------------------------
GITHUB_REPO = os.environ.get("OPENINSECT_REPO", "https://github.com/YoannDev90/OpenInsectID.git")
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

def _is_our_checkout(path):
    # Identify OUR repo precisely: any folder happening to contain a src/
    # (cwd leftovers, /kaggle, ...) must not be mistaken for the checkout.
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isdir(os.path.join(path, "src", "data_pipeline")))

# Walk up from cwd (kernels start in /kaggle/working, Jupyter in the notebook
# folder) to the repo root; anywhere else means: install from GitHub.
repo_root = os.getcwd()
while not _is_our_checkout(repo_root):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:
        break
    repo_root = parent
in_checkout = _is_our_checkout(repo_root)

if in_checkout:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
else:
    # pip needs the `git+` prefix to clone -- a plain https URL makes GitHub
    # serve the HTML page and pip fails with "cannot detect archive format".
    install_url = GITHUB_REPO
    if install_url.startswith(("http://", "https://")) and not install_url.startswith("git+"):
        install_url = "git+" + install_url
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_url])

try:
    import src.data_pipeline  # noqa: F401 - fail fast, with context, if shadowed
except ModuleNotFoundError:
    print("DIAG sys.path:", sys.path)
    raise

from pathlib import Path
import yaml

print(f"cwd={os.getcwd()} checkout={in_checkout} repo_root={repo_root}")
print("package ready; importable as `src.*`")


In [ ]:
# Export stack. onnx2tf (>=2.x) pins its own fully-py3.13-compatible stack
# (onnx, onnxruntime, onnxscript, ai-edge-litert), so a single install covers
# INT8 PTQ and the ONNX -> TFLite conversion; onnxscript also serves the
# dynamo exporter recent torch builds use by default.
import importlib.util, subprocess, sys

missing = [m for m in ("onnx", "onnxruntime", "onnxscript", "onnx2tf")
           if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnx2tf"], check=True
    )
else:
    print("export stack already present")


## Load the trained model

In [ ]:
import torch
from pathlib import Path
from src.models import HierarchicalMoE, TaxonomyTree

# Probe both Kaggle mount layouts, then fall back to a local run.
MOUNTS = [Path("/kaggle/input"), Path("/kaggle/input/datasets/yoanndev90")]
_ckpt_dir = next(
    (base / "open-insect-id-checkpoints"
     for base in MOUNTS if (base / "open-insect-id-checkpoints" / "moe_state.pt").exists()),
    Path("checkpoints"),
)
CKPT = str(_ckpt_dir / "moe_state.pt")
TAX = str(_ckpt_dir / "taxonomy.json")

tree = TaxonomyTree.load(TAX)
model = HierarchicalMoE(tree, backbone_name="mnv4_small")
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
model.eval()
print("loaded MoE:", model.n_orders, "orders /", model.n_species, "species")


## Export per-module ONNX

In [ ]:
from src.export import export_moe
from pathlib import Path

# 5 modules: trunk (image -> features) + one head per level (features ->
# global logits); manifest.json carries labels + child ranges for slicing.
onnx_dir = Path("export/onnx"); onnx_dir.mkdir(parents=True, exist_ok=True)
paths = export_moe(model, onnx_dir, image_size=224)
print(f"exported {len(paths)} modules -> {onnx_dir}")
print("manifest:", (onnx_dir / "manifest.json").exists())


## INT8 post-training quantization of the trunk (calibration set)

In [ ]:
from src.export import quantize_static_int8, quantize_dynamic_int8
import torch

# Calibration images: a representative batch from your validation set.
# (Only the trunk consumes images; the heads consume the trunk's features.)
calib = torch.randn(64, 3, 224, 224)

qdir = Path("export/onnx_int8"); qdir.mkdir(parents=True, exist_ok=True)
module_name = "trunk"
module_onnx = onnx_dir / f"{module_name}.onnx"
print("quantizing:", module_name)
q = quantize_static_int8(module_onnx, qdir / f"{module_name}_int8.onnx", calib)
print("quantized ->", q)


## Convert to TFLite

Uses the `onnx2tf` CLI (installed by the setup cell above: since v2.x it
pins its own py3.13-compatible stack — no tensorflow or tflite-support
needed). If it is absent the converter prints the exact install command
instead of failing.


In [ ]:
from src.export import TFLiteINT8Converter

conv = TFLiteINT8Converter()
tflite_dir = Path("export/tflite"); tflite_dir.mkdir(parents=True, exist_ok=True)
if conv.available():
    # Convert the trunk's INT8 (QDQ) ONNX: onnx2tf maps it to INT8 .tflite.
    tfl = conv.convert(qdir / f"{module_name}_int8.onnx", tflite_dir, int8=True)
    print("tflite ->", tfl)
else:
    print("onnx2tf not installed. Install with:")
    print("  !pip install onnx2tf")
    print("then re-run this cell.")


## Export the heads as FP16 matrices (option B)

The heads are pure `Linear`s: ship them as raw `(W, b)` matrices instead of
TFLite graphs — 4 tiny matmuls on-device, no head quantization error at 40k
logits, and no extra interpreters. Weights are exported per level as FP16.


In [ ]:
import numpy as np

heads_dir = Path("export/heads"); heads_dir.mkdir(parents=True, exist_ok=True)
for level in ("order", "family", "genus", "species"):
    head = getattr(model.heads, level)
    if head is None:
        continue  # single-order tree: no order head
    np.savez(
        heads_dir / f"head_{level}.npz",
        weight=head.weight.detach().cpu().numpy().astype(np.float16),
        bias=head.bias.detach().cpu().numpy().astype(np.float16),
    )
    print(f"head_{level}: W{tuple(head.weight.shape)} b{tuple(head.bias.shape)} -> fp16")
print("heads ->", heads_dir)


## Package the artifact

Bundle the INT8 trunk `.tflite`, the FP16 head matrices, `manifest.json`
(labels + child ranges for on-device slicing), and `taxonomy.json` into a
Kaggle dataset (or zip for GitHub Releases). On-device: run the trunk once,
then descend the cascade with 4 head matmuls.


In [ ]:
import shutil, json
bundle = Path("export/open_insect_id_bundle")
bundle.mkdir(parents=True, exist_ok=True)
for src in [Path("export/tflite"), Path("export/heads"), onnx_dir / "manifest.json"]:
    if src.exists():
        if src.is_dir():
            shutil.copytree(src, bundle / src.name, dirs_exist_ok=True)
        else:
            shutil.copy(src, bundle / src.name)
shutil.copy(TAX, bundle / "taxonomy.json")
shutil.make_archive("open_insect_id_tflite", "zip", root_dir=bundle)
print("bundle:", list(p.name for p in bundle.iterdir()))
